# MAVT
---
_**M**ulti-**A**ttribute **V**alue **T**heory_

In [1]:
import numpy as np
import pandas as pd

## 1. Introduction
---
**MAVT** (_Multi-Attribute Value Theory_) evaluates alternatives with **known consequences** using explicit functions that represent the decision maker's preferences. This recipe uses an **additive value model**, with single-attribute value functions and **swing weights**.

Value functions need not be linear in the original measurements. Their shape represents the strength of preference for improvements under certainty. For uncertain outcomes and preferences over lotteries, see [MAUT](./maut.ipynb).

### 1.1. Applications
---
The _MAVT approach_ can support decisions such as:

- Housing selection
- Training program selection
- Product design evaluation
- Public service planning

Consider **4 apartments** evaluated by **monthly rent** (BRL), **one-way commute** (minutes), and **usable area** (m²). All satisfy the household's essential requirements. Rent and commute are minimized; area is maximized.

The data and elicited preferences are **illustrative**. Consequences are treated as certain within this exercise; changes in travel time or future rent would require additional modeling.

In [2]:
df = pd.DataFrame({
    "rent": [2600, 3000, 2300, 2100],
    "commute": [20, 12, 45, 55],
    "area": [55, 42, 75, 60]
}, index=["Park flat", "Metro flat", "Garden flat", "Outer flat"])

df.style.background_gradient()

,rent,commute,area
Park flat,2600,20,55
Metro flat,3000,12,42
Garden flat,2300,45,75
Outer flat,2100,55,60


## 2. Method steps
---

### 2.1. Attributes, ranges and additive preferences
---
Agree on the relevant performance ranges before eliciting value functions and weights. The **worst reference bundle** is rent BRL 4,000, commute 60 minutes, and area 35 m²; the best bundle uses the opposite endpoints.

Assume the additive preference conditions are appropriate over these ranges: value differences and tradeoffs can be represented without interaction terms. Assess these conditions with the decision maker. Statistical independence of the observed attributes does not establish them.

In [3]:
OBJECTIVES = pd.Series(["min", "min", "max"], index=df.columns)
WORST = pd.Series([4000, 60, 35], index=df.columns)
BEST = pd.Series([1800, 10, 90], index=df.columns)

pd.DataFrame({"objective": OBJECTIVES, "worst": WORST, "best": BEST})

,objective,worst,best
rent,min,4000,1800
commute,min,60,10
area,max,35,90


### 2.2. Single-attribute value functions
---
Elicit value levels between **0** at the worst endpoint and **1** at the best endpoint. The table below gives the household's assessments, not scores inferred from the alternatives.

For area, improving from **35 to 50 m²** provides **0.60** of the full-range value gain; improving from **50 to 70 m²** adds **0.30**. This expresses diminishing value for additional space.

Between consecutive assessed points $(x_k,v_k)$ and $(x_{k+1},v_{k+1})$, use **linear interpolation**:

$$ \large
v_j(x)=v_k+\frac{x-x_k}{x_{k+1}-x_k}(v_{k+1}-v_k)
$$

In [4]:
VALUE_POINTS = {
    "rent": pd.Series([1.00, 0.80, 0.30, 0.00], index=[1800, 2400, 3200, 4000]),
    "commute": pd.Series([1.00, 0.85, 0.40, 0.00], index=[10, 20, 35, 60]),
    "area": pd.Series([0.00, 0.60, 0.90, 1.00], index=[35, 50, 70, 90])
}

pd.concat(VALUE_POINTS, names=["attribute", "performance"]).to_frame("value")

value
attribute performance       
rent      1800          1.00
          2400          0.80
          3200          0.30
          4000          0.00
commute   10            1.00
          20            0.85
          35            0.40
          60            0.00
area      35            0.00
          50            0.60
          70            0.90
          90            1.00

### 2.3. Evaluate the performances
---
Map each performance through its assessed function. Keep interpolation points in **increasing performance order**, including for cost attributes, whose values decrease. Reject out-of-range performances instead of silently clipping them.

In [5]:
df_v = pd.DataFrame(index=df.index)

for col, points in VALUE_POINTS.items():
    x = points.index.to_numpy(dtype=float)
    values = points.to_numpy(dtype=float)
    assert np.all(np.diff(x) > 0)
    direction = 1 if OBJECTIVES[col] == "max" else -1
    assert np.all(direction*np.diff(values) >= 0)
    if not df[col].between(x[0], x[-1]).all():
        raise ValueError("A performance is outside its assessed value function.")
    df_v[col] = np.interp(df[col], x, values)

df_v.style.background_gradient(vmin=0, vmax=1)

,rent,commute,area
Park flat,0.675000,0.850000,0.675000
Metro flat,0.425000,0.970000,0.280000
Garden flat,0.833333,0.240000,0.925000
Outer flat,0.900000,0.080000,0.750000


### 2.4. Swing weights
---
Starting from the **all-worst bundle**, compare improvements that move **one attribute through its full range**, holding the others at their worst levels. Assign **100 points** to the rent swing, **80** to the commute swing, and **60** to the area swing.

Normalize these assessed swing values:

$$ \large
w_j=\frac{p_j}{\sum_k p_k}, \qquad \sum_j w_j=1
$$

The weights belong to the stated ranges and value functions. Changing an endpoint requires reviewing the weights; generic statements about attribute importance are insufficient.

In [6]:
SWING_POINTS = pd.Series([100, 80, 60], index=df.columns, dtype=float)
WEIGHTS = SWING_POINTS/SWING_POINTS.sum()

pd.DataFrame({
    "starting level": WORST,
    "improved level": BEST,
    "swing points": SWING_POINTS,
    "weight": WEIGHTS
})

,starting level,improved level,swing points,weight
rent,4000,1800,100.0,0.416667
commute,60,10,80.0,0.333333
area,35,90,60.0,0.250000


### 2.5. Weighted contributions and overall value
---
Combine the single-attribute values in the additive model:

$$ \large
V(a_i)=\sum_{j=1}^n w_jv_j(x_{ij})
$$

Show each contribution to make the tradeoffs visible. The model permits compensation between attributes after the essential requirements have been satisfied.

In [7]:
df_contributions = df_v*WEIGHTS
df_contributions["score"] = df_contributions.sum(axis=1)

df_contributions

,rent,commute,area,score
Park flat,0.281250,0.283333,0.16875,0.733333
Metro flat,0.177083,0.323333,0.07000,0.570417
Garden flat,0.347222,0.080000,0.23125,0.658472
Outer flat,0.375000,0.026667,0.18750,0.589167


### 2.6. Score and Ranking
---
Rank alternatives by **decreasing overall value**. Equal values share a rank; no arbitrary tie-breaking is needed.

In [8]:
df_score = df.copy()
df_score["score"] = df_contributions["score"]
df_score["ranking"] = df_score["score"].rank(ascending=False, method="min").astype(int)

df_score.sort_values("ranking").style.background_gradient(subset=list(df.columns))

,rent,commute,area,score,ranking
Park flat,2600,20,55,0.733333,1
Garden flat,2300,45,75,0.658472,2
Outer flat,2100,55,60,0.589167,3
Metro flat,3000,12,42,0.570417,4


**Park flat** is preferred: its short commute and useful floor area justify the rent relative to the cheaper, more distant options. **Metro flat** has the shortest commute, but the household gives too little additional value to that small improvement to offset its rent and limited space.

### 2.7. Weight sensitivity
---
Vary the commute weight while preserving the **100:60** ratio between rent and area weights. Keep the value functions unchanged. These are alternative preference assumptions, not estimates of statistical uncertainty.

In [9]:
sensitivity = []

for commute_weight in [0.2, 0.4, 0.6, 0.8]:
    weights = pd.Series({
        "rent": (1 - commute_weight)*100/160,
        "commute": commute_weight,
        "area": (1 - commute_weight)*60/160
    })
    scores = df_v.mul(weights).sum(axis=1)
    sensitivity.append([commute_weight, scores.idxmax(), scores.max()])

pd.DataFrame(sensitivity, columns=["commute weight", "preferred apartment", "value"])

,commute weight,preferred apartment,value
0,0.2,Garden flat,0.742167
1,0.4,Park flat,0.745000
2,0.6,Park flat,0.780000
3,0.8,Metro flat,0.850125


With a commute weight of **0.20**, **Garden flat** leads because rent and area receive more weight. The preference changes to **Metro flat** when the commute swing receives weight **0.80**. This makes the recommendation's dependence on the household's tradeoffs explicit.

## 3. Examples
---

### 3.1. Staff training program
---
A company compares **3 training programs** meeting the same learning objectives. Evaluate **fee** (BRL per employee, minimize), **supervised practice** (hours, maximize), and **absence from regular work** (days, minimize).

The data and preferences are **illustrative** and treated as certain. The manager assesses the following piecewise value functions and full-range swings, assigning **100 points** to practice, **60** to fee, and **40** to absence.

In [10]:
df_ex1 = pd.DataFrame({
    "fee": [1800, 2500, 900],
    "practice": [24, 12, 8],
    "absence": [3, 5, 1]
}, index=["Simulation workshop", "Certification course", "Self-paced modules"])

df_ex1.style.background_gradient()

,fee,practice,absence
Simulation workshop,1800,24,3
Certification course,2500,12,5
Self-paced modules,900,8,1


In [11]:
OBJECTIVES_ex1 = pd.Series(["min", "max", "min"], index=df_ex1.columns)
VALUE_POINTS_ex1 = {
    "fee": pd.Series([1.00, 0.85, 0.50, 0.00], index=[0, 1000, 2000, 3000]),
    "practice": pd.Series([0.00, 0.50, 0.80, 1.00], index=[0, 8, 16, 24]),
    "absence": pd.Series([1.00, 0.85, 0.40, 0.00], index=[0, 2, 5, 10])
}
SWING_POINTS_ex1 = pd.Series([60, 100, 40], index=df_ex1.columns, dtype=float)
WEIGHTS_ex1 = SWING_POINTS_ex1/SWING_POINTS_ex1.sum()

pd.concat(VALUE_POINTS_ex1, names=["attribute", "performance"]).to_frame("value")

value
attribute performance       
fee       0             1.00
          1000          0.85
          2000          0.50
          3000          0.00
practice  0             0.00
          8             0.50
          16            0.80
          24            1.00
absence   0             1.00
          2             0.85
          5             0.40
          10            0.00

In [12]:
df_ex1_v = pd.DataFrame(index=df_ex1.index)

for col, points in VALUE_POINTS_ex1.items():
    x = points.index.to_numpy(dtype=float)
    values = points.to_numpy(dtype=float)
    if not df_ex1[col].between(x[0], x[-1]).all():
        raise ValueError("A performance is outside its assessed value function.")
    df_ex1_v[col] = np.interp(df_ex1[col], x, values)

df_ex1_v.style.background_gradient(vmin=0, vmax=1)

,fee,practice,absence
Simulation workshop,0.570000,1.000000,0.700000
Certification course,0.250000,0.650000,0.400000
Self-paced modules,0.865000,0.500000,0.925000


In [13]:
WEIGHTS_ex1.to_frame("weight")

,weight
fee,0.3
practice,0.5
absence,0.2


In [14]:
df_ex1_contributions = df_ex1_v*WEIGHTS_ex1
df_ex1_contributions["score"] = df_ex1_contributions.sum(axis=1)

df_ex1_contributions

,fee,practice,absence,score
Simulation workshop,0.1710,0.500,0.140,0.8110
Certification course,0.0750,0.325,0.080,0.4800
Self-paced modules,0.2595,0.250,0.185,0.6945


In [15]:
df_ex1_score = df_ex1.copy()
df_ex1_score["score"] = df_ex1_contributions["score"]
df_ex1_score["ranking"] = (
    df_ex1_score["score"].rank(ascending=False, method="min").astype(int)
)

df_ex1_score.sort_values("ranking").style.background_gradient(subset=list(df_ex1.columns))

,fee,practice,absence,score,ranking
Simulation workshop,1800,24,3,0.811000,1
Self-paced modules,900,8,1,0.694500,2
Certification course,2500,12,5,0.480000,3


The **simulation workshop** leads because its extensive supervised practice outweighs the extra fee and absence relative to self-paced modules. The certification course is less attractive on all three modeled attributes than the workshop. This illustrates both tradeoffs and the model's respect for dominance.

## 4. References
---

- Dyer, J. S. and Sarin, R. K. (1979). [*Measurable Multiattribute Value Functions*](https://doi.org/10.1287/opre.27.4.810).
- Keeney, R. L. and Raiffa, H. (1993). [*Decisions with Multiple Objectives: Preferences and Value Trade-Offs*](https://doi.org/10.1017/CBO9781139174084). Chapter 3 treats tradeoffs under certainty; first published in 1976.
- Parnell, G. S. (2009). [*Decision Analysis in One Chart*](https://extapps.ksc.nasa.gov/reliability/Documents/Decision_Analysis_in_One_Chart.pdf). Describes value functions and swing weighting.